# CO2 Corrosion Analysis with Electrolyte CPA

This notebook demonstrates NeqSim's `CO2CorrosionAnalyzer` — a coupled thermodynamic–corrosion
model that uses the **electrolyte CPA equation of state** for rigorous pH prediction and the
**de Waard-Milliams (1991)** model for CO2 corrosion rate estimation.

**Key features:**
- Rigorous pH from aqueous H3O+ activity (chemical reaction equilibrium)
- CO2 partial pressure from thermodynamic flash
- Temperature and pressure sweep capabilities
- Corrosion severity classification per NORSOK M-001
- FeCO3 scale formation prediction

**Applicable standards:** NORSOK M-506, NACE SP0775

In [ ]:
# Setup — load NeqSim
import subprocess, sys
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim'])

from neqsim import jneqsim
import jpype

# Import the CO2CorrosionAnalyzer
CO2CorrosionAnalyzer = jpype.JClass('neqsim.pvtsimulation.flowassurance.CO2CorrosionAnalyzer')

import matplotlib.pyplot as plt
import numpy as np
print('NeqSim loaded successfully')

## 1. Single-Point Analysis

Analyze CO2 corrosion for a CCS pipeline: 95% CO2 + 5% water at 60°C and 100 bar.

In [ ]:
# Create analyzer for typical CCS conditions
analyzer = CO2CorrosionAnalyzer(60.0, 100.0)
analyzer.setCO2MoleFractionInGas(0.95)
analyzer.setWaterMoleFractionInGas(0.05)
analyzer.setFlowVelocity(3.0)       # m/s
analyzer.setPipeDiameter(0.254)      # 10-inch pipe
analyzer.run()

print('=== CO2 Corrosion Analysis (CCS Pipeline) ===')
print(f'Temperature:           {60.0:.1f} °C')
print(f'Pressure:              {100.0:.1f} bara')
print(f'CO2 partial pressure:  {float(analyzer.getCO2PartialPressure()):.1f} bar')
print(f'Free water present:    {bool(analyzer.isFreeWaterPresent())}')
print(f'Aqueous pH (eCPA):     {float(analyzer.getAqueousPH()):.2f}')
print(f'Baseline rate:         {float(analyzer.getBaselineCorrosionRate()):.2f} mm/yr')
print(f'Corrected rate:        {float(analyzer.getCorrosionRate()):.2f} mm/yr')
print(f'Severity:              {str(analyzer.getCorrosionSeverity())}')
print(f'25-yr allowance:       {float(analyzer.estimateCorrosionAllowance(25.0)):.1f} mm')

## 2. Temperature Sweep — Corrosion Rate Profile

The de Waard-Milliams model predicts increasing corrosion with temperature up to ~80°C,
then FeCO3 scale formation reduces the rate at higher temperatures.

In [ ]:
# Temperature sweep: 10°C to 100°C
analyzer_sweep = CO2CorrosionAnalyzer(40.0, 50.0)
analyzer_sweep.setCO2MoleFractionInGas(0.50)
analyzer_sweep.setWaterMoleFractionInGas(0.50)

results = analyzer_sweep.runTemperatureSweep(10.0, 100.0, 18)

temperatures = [float(r.get('temperature_C')) for r in results]
pH_values = [float(r.get('pH')) for r in results]
corrosion_rates = [float(r.get('corrosionRate_mmyr')) for r in results]
baseline_rates = [float(r.get('baselineRate_mmyr')) for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Corrosion rate vs temperature
ax1.plot(temperatures, baseline_rates, 'b--', label='Baseline (no corrections)', linewidth=1.5)
ax1.plot(temperatures, corrosion_rates, 'r-o', label='Corrected rate', linewidth=2, markersize=4)
ax1.set_xlabel('Temperature (°C)', fontsize=12)
ax1.set_ylabel('Corrosion Rate (mm/yr)', fontsize=12)
ax1.set_title('CO2 Corrosion Rate vs Temperature', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0.1, color='green', linestyle=':', alpha=0.5, label='Low/Medium boundary')
ax1.axhline(y=1.0, color='orange', linestyle=':', alpha=0.5, label='High/Very High boundary')

# pH vs temperature
ax2.plot(temperatures, pH_values, 'g-s', linewidth=2, markersize=4)
ax2.set_xlabel('Temperature (°C)', fontsize=12)
ax2.set_ylabel('pH (electrolyte CPA)', fontsize=12)
ax2.set_title('Aqueous Phase pH vs Temperature', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('corrosion_temperature_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Temperature range: {temperatures[0]:.0f}–{temperatures[-1]:.0f} °C')
print(f'pH range: {min(pH_values):.2f} – {max(pH_values):.2f}')
print(f'Max corrosion rate: {max(corrosion_rates):.2f} mm/yr')

## 3. Pressure Sweep — Effect of CO2 Partial Pressure

Higher pressure means higher CO2 partial pressure (more CO2 dissolves in water),
leading to lower pH and higher corrosion rates.

In [ ]:
# Pressure sweep: 10 to 200 bar at 60°C
analyzer_p = CO2CorrosionAnalyzer(60.0, 50.0)
analyzer_p.setCO2MoleFractionInGas(0.50)
analyzer_p.setWaterMoleFractionInGas(0.50)

results_p = analyzer_p.runPressureSweep(10.0, 200.0, 19)

pressures = [float(r.get('pressure_bara')) for r in results_p]
pH_p = [float(r.get('pH')) for r in results_p]
rate_p = [float(r.get('corrosionRate_mmyr')) for r in results_p]
pCO2_p = [float(r.get('co2PartialPressure_bar')) for r in results_p]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(pressures, rate_p, 'r-o', linewidth=2, markersize=4)
ax1.set_xlabel('Total Pressure (bara)', fontsize=12)
ax1.set_ylabel('Corrosion Rate (mm/yr)', fontsize=12)
ax1.set_title('CO2 Corrosion Rate vs Pressure (60°C)', fontsize=13)
ax1.grid(True, alpha=0.3)

ax2.plot(pressures, pH_p, 'g-s', linewidth=2, markersize=4)
ax2.set_xlabel('Total Pressure (bara)', fontsize=12)
ax2.set_ylabel('pH (electrolyte CPA)', fontsize=12)
ax2.set_title('Aqueous Phase pH vs Pressure', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('corrosion_pressure_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Effect of Corrosion Inhibitor

Chemical inhibitors reduce corrosion rate. Typical efficiencies: 70–90%.

In [ ]:
# Compare uninhibited vs inhibited corrosion
inhibitor_efficiencies = [0.0, 0.50, 0.70, 0.80, 0.90, 0.95]
rates_inh = []

for eff in inhibitor_efficiencies:
    a = CO2CorrosionAnalyzer(60.0, 100.0)
    a.setCO2MoleFractionInGas(0.95)
    a.setWaterMoleFractionInGas(0.05)
    a.setInhibitorEfficiency(eff)
    a.run()
    rates_inh.append(float(a.getCorrosionRate()))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([f'{e*100:.0f}%' for e in inhibitor_efficiencies], rates_inh,
              color=['#d32f2f', '#f57c00', '#fbc02d', '#388e3c', '#1976d2', '#7b1fa2'])
ax.set_xlabel('Inhibitor Efficiency', fontsize=12)
ax.set_ylabel('Corrosion Rate (mm/yr)', fontsize=12)
ax.set_title('Effect of Corrosion Inhibitor on CO2 Corrosion', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, rate in zip(bars, rates_inh):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            f'{rate:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('corrosion_inhibitor_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Uninhibited rate: {rates_inh[0]:.2f} mm/yr')
print(f'90% inhibitor:    {rates_inh[4]:.2f} mm/yr (reduction: {(1-rates_inh[4]/rates_inh[0])*100:.0f}%)')

## 5. Natural Gas with CO2 — Lower CO2 Content

Typical natural gas pipeline corrosion scenario: 5% CO2 in methane at 80 bar.

In [ ]:
# Natural gas scenario
ng_analyzer = CO2CorrosionAnalyzer(40.0, 80.0)
ng_analyzer.setCO2MoleFractionInGas(0.05)
ng_analyzer.setWaterMoleFractionInGas(0.02)
ng_analyzer.setMethaneMoleFractionInGas(0.93)
ng_analyzer.run()

print('=== Natural Gas Pipeline Corrosion ===')
print(f'CO2 partial pressure:  {float(ng_analyzer.getCO2PartialPressure()):.1f} bar')
print(f'Free water:            {bool(ng_analyzer.isFreeWaterPresent())}')
print(f'pH:                    {float(ng_analyzer.getAqueousPH()):.2f}')
print(f'Corrosion rate:        {float(ng_analyzer.getCorrosionRate()):.2f} mm/yr')
print(f'Severity:              {str(ng_analyzer.getCorrosionSeverity())}')

## 6. JSON Report Output

The analyzer produces a comprehensive JSON report with all inputs, flash results,
and corrosion predictions.

In [ ]:
import json

# Get JSON report from the CCS analyzer
json_str = str(analyzer.toJson())
report = json.loads(json_str)

print(json.dumps(report, indent=2))